# **Unstructured RAG**
Retrieves relevant information from unstructured data such as PDFs, documents, and text files using semantic search.
The retrieved information is given to an LLM as context to generate a relevant and grounded answer.

### **Components & Architecture**
- **Document Parsing:** `unstructured.io` (Text, Table, Image extraction)
- **Embedding Model:** `OpenAIEmbeddings`
- **Vector Database:** FAISS (Local)
- **LLM Model:** `ChatOpenAI`

## **Initial Setup**

In [ ]:
!apt-get install poppler-utils
!apt-get install tesseract-ocr
!apt-get install libtesseract-dev

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

## **Indexing**

In [ ]:
# load embedding model
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [ ]:
 # load and extract images, tables, and chunk text
from unstructured.partition.pdf import partition_pdf

filename = "/content/sample.pdf"

pdf_elements = partition_pdf(
    filename=filename,
    extract_images_in_pdf=True,
    strategy = "hi_res",
    hi_res_model_name="yolox",
    infer_table_structure=True,
    chunking_strategy="by_title",
    max_characters=3000,
    combine_text_under_n_chars=200,
)

In [ ]:
# check unique categories
from collections import Counter
category_counts = Counter(str(type(element)) for element in pdf_elements)
unique_categories = set(category_counts)
category_counts

In [ ]:
# extract unique types
unique_types = {el.to_dict()['type'] for el in pdf_elements}
unique_types

In [ ]:
# # display images from pdf
# from IPython.display import Image, display
# image_files = os.listdir('/content/figures')
# image_files = [os.path.join('/content/figures', image_file) for image_file in image_files]

# for image_file in image_files:
#     display(Image(filename=image_file))

In [ ]:
# convert pdf_elements to langchain documents
from langchain.schema import Document
documents = [Document(page_content=el.text, metadata={"source": filename}) for el in pdf_elements]

## **Vector Store**

In [ ]:
# create vectorstore
from langchain.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents, embeddings)

## **Retriever**

In [ ]:
# create retriever
retriever = vectorstore.as_retriever()

## **RAG Chain**

In [ ]:
# load llm
from langchain_openai import ChatOpenAI
llm = ChatOpenAI()

In [ ]:
# create document chain
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

template = """"
You are a helpful assistant that answers questions based on the provided context, which can include text and tables.
Use the provided context to answer the question.
Question: {input}
Context: {context}
Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

# Setup RAG pipeline
rag_chain = (
    {"context": retriever,  "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# response
response = rag_chain.invoke("Compare all the Training Results on MATH Test Set")
response